In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import html
import random
import pandas as pd
from pathlib import Path
import os
from src.utils.config import find_project_root, load_config

In [2]:
# =============================================================================
# SECTION 0: Configuration
# =============================================================================

# Determine root directory of project and load configuration file
project_root = find_project_root()
config = load_config()

# Get current working directory 
pwd = os.getcwd()

# Specify path to loan-level address data
address_dir = os.path.join(pwd,'geocoding_input')

# Path to the input data file (pre-filtered to disagreements only).
# Supported formats: .parquet or .csv
INPUT_DATA_PATH = os.path.join(address_dir,'manual_review_parsed_addresses.parquet')

# Folder where review_results.parquet will be saved.
# Created automatically if it does not exist.
OUTPUT_FOLDER = os.path.join(pwd,'manual_review/inconsistent_address_parsing')

# Number of cases to review per session.
BATCH_SIZE = 10

In [3]:
# =============================================================================
# SECTION 1: Load input data
# =============================================================================

input_path = Path(INPUT_DATA_PATH)

if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path.resolve()}")

if input_path.suffix == ".parquet":
    df = pd.read_parquet(input_path)
elif input_path.suffix == ".csv":
    df = pd.read_csv(input_path)
else:
    raise ValueError(
        f"Unsupported format: '{input_path.suffix}'. Please provide a .parquet or .csv file."
    )

print(f"Loaded {len(df)} row(s) from: {input_path.resolve()}")

# =============================================================================
# SECTION 2: Load existing results and identify previously processed cases
# =============================================================================

output_folder = Path(OUTPUT_FOLDER)
output_folder.mkdir(parents=True, exist_ok=True)
output_path = output_folder / "review_results.parquet"

if output_path.exists():
    existing_results_df = pd.read_parquet(output_path)
    processed_ids = set(existing_results_df["masterloanidtrepp"].tolist())
    print(f"Found existing results: {len(existing_results_df)} case(s) already reviewed.")
else:
    existing_results_df = pd.DataFrame()
    processed_ids = set()
    print("No existing results found. Starting fresh.")

# =============================================================================
# SECTION 3: Build the unprocessed case list, then apply the batch limit
# =============================================================================

all_unprocessed = []

for loan_id, group in df.groupby("masterloanidtrepp"):
    if loan_id in processed_ids:
        continue  # skip cases reviewed in prior sessions
    rows = group.reset_index(drop=True)
    if len(rows) != 2:
        print(f"Warning: loan ID {loan_id} has {len(rows)} row(s) — expected 2. Skipping.")
        continue
    all_unprocessed.append((loan_id, rows))

# Slice to the current batch
case_list = all_unprocessed[:BATCH_SIZE]

# remaining_after_batch is captured by the _advance() closure below;
# it reflects how many cases will be left after this session completes.
remaining_after_batch = len(all_unprocessed) - len(case_list)

if not case_list:
    print("\n✅ All cases have already been reviewed. Nothing to do.")
else:
    print(
        f"\n{len(all_unprocessed)} unprocessed case(s) found.\n"
        f"This session: reviewing {len(case_list)} case(s). "
        f"{remaining_after_batch} case(s) will remain after this batch."
    )

# =============================================================================
# SECTION 4: Schema validation
# =============================================================================

_VALID_ADDRESS_TYPES = {"exact", "range", "approximate"}


def validate_address_components(parsed):
    """
    Validate a parsed address_components value against the required schema.
    Returns a list of error strings. An empty list means the input is valid.

    Schema rules per element:
      address_type           : "exact" | "range" | "approximate"
      first_building_number  : integer when address_type is "exact"/"range";
                               null    when address_type is "approximate"
      last_building_number   : same rules as first_building_number
      street                 : non-empty string
    """
    errors = []

    # --- Top-level structure ---
    if not isinstance(parsed, list):
        return ["Top-level value must be a JSON array, e.g. [ { ... } ]."]
    if len(parsed) == 0:
        return ["The array must contain at least one element."]

    for i, element in enumerate(parsed):
        label = f"Element {i + 1}"

        if not isinstance(element, dict):
            errors.append(f"{label}: expected a JSON object, got {type(element).__name__}.")
            continue  # can't check individual fields without a dict

        # --- Required and unexpected keys ---
        required_keys   = {"address_type", "first_building_number",
                           "last_building_number", "street"}
        present_keys    = set(element.keys())
        missing_keys    = required_keys - present_keys
        unexpected_keys = present_keys - required_keys

        if missing_keys:
            errors.append(
                f"{label}: missing required key(s): "
                f"{', '.join(repr(k) for k in sorted(missing_keys))}."
            )
        if unexpected_keys:
            errors.append(
                f"{label}: unexpected key(s): "
                f"{', '.join(repr(k) for k in sorted(unexpected_keys))}."
            )

        # --- address_type ---
        addr_type = element.get("address_type")
        if "address_type" in element:
            if addr_type not in _VALID_ADDRESS_TYPES:
                errors.append(
                    f"{label}: 'address_type' must be one of "
                    f"{sorted(_VALID_ADDRESS_TYPES)}, got {repr(addr_type)}."
                )

        # --- Building numbers ---
        # Both fields share identical validation logic, so we loop over them.
        for field in ("first_building_number", "last_building_number"):
            if field not in element:
                continue  # missing key already reported above

            val = element[field]

            if addr_type == "approximate":
                # Must be null for approximate addresses
                if val is not None:
                    errors.append(
                        f"{label}: '{field}' must be null when "
                        f"address_type is 'approximate' (got {repr(val)})."
                    )
            else:
                # Must be a non-null integer for exact/range addresses.
                # We explicitly exclude bool because in Python bool is a
                # subclass of int, so isinstance(True, int) is True.
                if val is None:
                    errors.append(
                        f"{label}: '{field}' must be an integer when "
                        f"address_type is '{addr_type}' (null is not allowed)."
                    )
                elif isinstance(val, bool) or not isinstance(val, int):
                    errors.append(
                        f"{label}: '{field}' must be an integer, "
                        f"got {type(val).__name__} ({repr(val)})."
                    )

        # --- street ---
        if "street" in element:
            street = element["street"]
            if not isinstance(street, str) or not street.strip():
                errors.append(
                    f"{label}: 'street' must be a non-empty string "
                    f"(got {repr(street)})."
                )

    return errors


# =============================================================================
# SECTION 5: Session state
# =============================================================================

# Accumulates results recorded during this session.
# Merged with existing_results_df and saved to parquet at the end of the batch.
session_results = []

# Mutable dict used to share state between event handlers.
# A dict is used (rather than plain variables) because Python closures
# do not allow rebinding of outer-scope names inside nested functions.
state = {
    "current_index": 0,
    "assignment":    None,   # maps 'A'/'B' -> the corresponding DataFrame row (Series)
}

# =============================================================================
# SECTION 6: Create widgets
# =============================================================================

progress_label = widgets.HTML(value="")
info_html      = widgets.HTML(value="")
option_a_html  = widgets.HTML(value="")
option_b_html  = widgets.HTML(value="")

selection_radio = widgets.RadioButtons(
    options=["Option A", "Option B", "Neither — enter custom response"],
    value="Option A",
    description="Your choice:",
    style={"description_width": "initial"},   # prevents label truncation
    layout=widgets.Layout(width="auto"),
)

custom_textarea = widgets.Textarea(
    placeholder=(
        'Enter a valid JSON array, e.g.\n'
        '[{"address_type": "exact", "first_building_number": 100, '
        '"last_building_number": 100, "street": "Main Street"}]'
    ),
    layout=widgets.Layout(width="100%", height="150px"),
)

custom_instructions_html = widgets.HTML(
    value=(
        '<span style="color:#555; font-size:12px;">'
        "Option A's output has been pre-loaded as a starting template — edit as needed. "
        "Input is validated against the required schema on submission."
        "</span>"
    )
)

# Group instructions and textarea into one container so we can
# show/hide both with a single layout toggle.
custom_section = widgets.VBox(
    [custom_instructions_html, custom_textarea],
    layout=widgets.Layout(display="none", margin="6px 0 0 0"),
)

# Output widget for validation error messages; using Output lets us
# call print() inside it and clear it cleanly between submissions.
feedback_output = widgets.Output()

submit_button = widgets.Button(
    description="Submit & Next →",
    button_style="primary",
    icon="check",
    layout=widgets.Layout(width="185px"),
)

completion_output = widgets.Output()

# =============================================================================
# SECTION 7: Helper functions
# =============================================================================

def parse_components(value):
    """
    Convert address_components to a Python object.
    Handles both JSON-encoded strings (common when loading from CSV/parquet)
    and already-parsed Python lists or dicts.
    """
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value    # return the raw string rather than crashing
    return value


def format_components_html(value, label):
    """
    Render address_components as a styled HTML block.
    html.escape() prevents special characters (< > & " ')
    from breaking the surrounding HTML structure.
    """
    parsed = parse_components(value)
    try:
        pretty = json.dumps(parsed, indent=2)
    except TypeError:
        pretty = str(parsed)

    escaped = html.escape(pretty)

    return (
        f'<div style="margin-bottom:8px;">'
        f"<b>Option {label}</b>"
        f'<pre style="background:#f8f8f8; border:1px solid #ddd; '
        f'border-radius:4px; padding:8px; font-size:12px; '
        f'white-space:pre-wrap; word-break:break-word;">'
        f"{escaped}</pre></div>"
    )


def load_case(index):
    """
    Populate all widgets with data for the case at the given index.
    Randomly assigns model outputs to Option A / B to blind the reviewer
    to which model produced which result.
    """
    loan_id, rows = case_list[index]

    # Shuffle to randomise which model appears as A vs. B
    order = [0, 1]
    random.shuffle(order)
    state["assignment"] = {
        "A": rows.iloc[order[0]],
        "B": rows.iloc[order[1]],
    }

    progress_label.value = (
        f"<b>Case {index + 1} of {len(case_list)}</b> "
        f'<span style="color:#888; font-weight:normal;">'
        f"({len(session_results)} submitted this session)</span>"
    )

    address_str = rows.iloc[0]["address_string"]
    info_html.value = (
        f'<div style="margin:8px 0;">'
        f"<b>Loan ID:</b> {loan_id}<br>"
        f'<b>Address string:</b> <span style="font-family:monospace;">'
        f"{html.escape(str(address_str))}</span>"
        f"</div>"
    )

    option_a_html.value = format_components_html(
        state["assignment"]["A"]["address_components"], label="A"
    )
    option_b_html.value = format_components_html(
        state["assignment"]["B"]["address_components"], label="B"
    )

    # Reset controls for each new case
    selection_radio.value = "Option A"
    custom_textarea.value = ""
    custom_section.layout.display = "none"
    feedback_output.clear_output()


def save_results():
    """
    Merge session results with any previously saved results and write
    the combined dataset to output_path as a parquet file.
    Returns the total number of rows saved.
    """
    session_df = pd.DataFrame(session_results)

    # existing_results_df was loaded at cell startup (Section 2);
    # it holds all results from prior sessions.
    if not existing_results_df.empty:
        combined_df = pd.concat([existing_results_df, session_df], ignore_index=True)
    else:
        combined_df = session_df

    combined_df.to_parquet(output_path, index=False)
    return len(combined_df)


# =============================================================================
# SECTION 8: Event handlers
# =============================================================================

def on_radio_change(change):
    """
    Show or hide the custom textarea based on the radio selection.
    Pre-populates the textarea with Option A's output as an editable template
    whenever 'Neither' is selected.
    """
    if change["new"] == "Neither — enter custom response":
        a_parsed = parse_components(state["assignment"]["A"]["address_components"])
        try:
            custom_textarea.value = json.dumps(a_parsed, indent=2)
        except TypeError:
            custom_textarea.value = ""
        custom_section.layout.display = ""       # reveal the section
    else:
        custom_section.layout.display = "none"   # hide the section


def on_submit(button):
    """
    Handle a click on the Submit button.

    For model-output selections (A or B):
      Record the result and advance immediately.

    For custom input ('Neither'):
      1. Check JSON syntax.
      2. Validate against the address_components schema.
      Record only if both checks pass; otherwise display error messages.
    """
    feedback_output.clear_output()

    index         = state["current_index"]
    loan_id, rows = case_list[index]
    address_str   = rows.iloc[0]["address_string"]
    chosen        = selection_radio.value

    if chosen in ("Option A", "Option B"):
        label = "A" if chosen == "Option A" else "B"
        row   = state["assignment"][label]

        session_results.append({
            "masterloanidtrepp":  loan_id,
            "address_string":     address_str,
            "selected_model":     row["model_id"],
            # Re-serialise to a canonical JSON string for consistent parquet storage
            "address_components": json.dumps(
                parse_components(row["address_components"])
            ),
            "selection_type": "model_output",
        })
        _advance()

    else:   # "Neither — enter custom response"
        raw = custom_textarea.value.strip()

        # --- Step 1: JSON syntax ---
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError as exc:
            with feedback_output:
                print(f"⚠️  JSON syntax error — {exc}")
                print("Please correct the JSON and click Submit again.")
            return

        # --- Step 2: Schema validation ---
        schema_errors = validate_address_components(parsed)
        if schema_errors:
            with feedback_output:
                print("⚠️  Schema validation failed:")
                for err in schema_errors:
                    print(f"   • {err}")
                print("\nPlease fix the issue(s) above and click Submit again.")
            return

        # Both checks passed — record the result
        session_results.append({
            "masterloanidtrepp":  loan_id,
            "address_string":     address_str,
            "selected_model":     "manual",
            "address_components": json.dumps(parsed),   # canonical JSON string
            "selection_type":     "manual",
        })
        _advance()


def _advance():
    """
    Move to the next case. When the batch is complete, save all results
    to parquet and display a summary.
    """
    state["current_index"] += 1

    if state["current_index"] < len(case_list):
        load_case(state["current_index"])

    else:
        # ---- Batch complete ----

        # Disable all controls to prevent accidental changes
        submit_button.disabled   = True
        selection_radio.disabled = True
        custom_textarea.disabled = True

        # Save and summarise
        total_saved = save_results()
        n_session = len(session_results)
        n_manual  = sum(1 for r in session_results if r["selection_type"] == "manual")

        with completion_output:
            clear_output()
            print("✅  Batch complete!\n")
            print(f"   This session : {n_session} case(s) reviewed "
                  f"({n_manual} manual, {n_session - n_manual} model output)")
            print(f"   Total saved  : {total_saved} case(s)")
            print(f"   Output file  : {output_path.resolve()}\n")
            if remaining_after_batch > 0:
                print(
                    f"   ℹ️  {remaining_after_batch} case(s) remain. "
                    "Re-run Cell 2 to start the next batch."
                )
            else:
                print("   🎉 All cases have been reviewed!")


# =============================================================================
# SECTION 9: Wire up event handlers
# =============================================================================

selection_radio.observe(on_radio_change, names="value")
submit_button.on_click(on_submit)

# =============================================================================
# SECTION 10: Assemble and launch the UI
# =============================================================================

ui = widgets.VBox([
    progress_label,
    widgets.HTML('<hr style="margin:4px 0;">'),
    info_html,
    option_a_html,
    option_b_html,
    widgets.HTML('<hr style="margin:4px 0;">'),
    selection_radio,
    custom_section,
    feedback_output,
    submit_button,
    completion_output,
], layout=widgets.Layout(max_width="800px", padding="12px"))

# Only launch the UI if there are cases to review.
# If there are none, the message was already printed in Section 3.
if case_list:
    load_case(0)
    display(ui)

Loaded 2666 row(s) from: /proj/characklab/projects/kieranf/CMBS/analysis/geocoding/geocoding_input/manual_review_parsed_addresses.parquet
Found existing results: 30 case(s) already reviewed.

1303 unprocessed case(s) found.
This session: reviewing 10 case(s). 1293 case(s) will remain after this batch.
